## Clinical Evidence evals: faithfulness, precision and relevance

Runs the same metrics as the MoA evals (`moa_innovation/evals`) against the cached output of `clinical_evidence_scoring`.

| Metric | What it measures | Module | Method |
|---|---|---|---|
| **Score check** | P, D, G, E, C are in range and the final score equals `min(5, max(1, (P + mode(D,G,E))/2 + C))` | `score_check` | Deterministic |
| **Claim precision** (faithfulness) | Claims supported by the retrieved web sources or the pipeline's own trial data, divided by ALL extracted claims. Unverifiable and errored claims count against the score. | `claim_verification` | Claude fetches the saved source pages |
| **Judge faithfulness** | Is the assigned score supported by the text, rationale and recorded per-component reasoning, and is it internally consistent | `llm_judge` | LLM judge against the rubric |
| **Context precision** | Share of the retrieved sources that are relevant to this drug's clinical evidence (also per RWE source) | `context_precision` | LLM judge over saved sources |
| **Field relevance** | Does each judged text field (RWE synthesis, consistency report, evidence gaps, rationale) answer its own question for this drug (1-5) | `relevance_judge` | LLM judge, per-field rubric |
| **Claim relevance** | Share of claims that bear on the evidence score (direct = 1, contextual = 0.5, irrelevant = 0) | `relevance_judge` | LLM judge |

The composite is a weighted average of the components that ran (`scoring.DEFAULT_WEIGHTS`, shared with the MoA evals). Components that failed or were skipped are dropped and the weights renormalised; `composite_coverage` shows how much weight was used, and no status is given below `MIN_COVERAGE`.

Input is the `output_payload.json` cached in GCS by `clinical_evidence_scoring`. Grounding sources are **not** taken from the payload: they are read only from `clinical_evidence/sources/<molecule>/sources_*.json` (the file closest in time to the payload). If that folder has no file for the molecule, claim verification checks claims against the pipeline data only and context precision reports an error.


In [ ]:
from pathlib import Path
import sys

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "medical_potential" / "config.py").is_file()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from medical_potential.clinical_evidence.evals.data import (
    load_rows_from_payload_files,
    load_clinical_evidence_rows,
)

SELECTED_MOLECULES = [
    "Bofanglutide",
]

# Latest cached payload per molecule from GCS (one row per molecule).
clinical_evidence_df = load_clinical_evidence_rows(SELECTED_MOLECULES)

# Offline alternative: evaluate payload JSON files on disk instead of GCS.
# clinical_evidence_df = load_rows_from_payload_files([
#     repo_root / "notebooks" / "input" / "rationale" / "Bofanglutide" / "clinical_evidence_payload.json",
# ])

clinical_evidence_df[["drug_name", "created_at", "score", "total_trials", "cache_gcs_uri"]]

In [ ]:
from medical_potential.config import CLAUDE_SONNET_MODEL
from medical_potential.claude_utils import get_claude_call_stats, reset_claude_call_stats
from medical_potential.clinical_evidence.evals.bq_push import build_bq_rows, push_eval_rows
from medical_potential.clinical_evidence.evals.claim_verification import (
    run_claim_verification,
    summarize_claims,
)
from medical_potential.clinical_evidence.evals.score_check import (
    check_clinical_evidence_score,
    summarize_score_check,
)
from medical_potential.clinical_evidence.evals.context_precision import run_context_precision
from medical_potential.clinical_evidence.evals.llm_judge import run_llm_judge
from medical_potential.clinical_evidence.evals.relevance_judge import (
    run_claim_relevance,
    run_field_relevance,
    summarize_claim_relevance,
)
from medical_potential.clinical_evidence.evals.scoring import DEFAULT_WEIGHTS, build_eval_scores
from medical_potential.clinical_evidence.evals.utils import eval_timestamp, latest_rows

EVAL_MODEL = CLAUDE_SONNET_MODEL  # Claude on Vertex AI
EVAL_RUN_ID = eval_timestamp()
PUSH_TO_BQ = False

FETCH_SOURCE_PAGES_FOR_PRECISION = False  # True lets Claude open every source (slower, costlier)
SCORE_WEIGHTS = dict(DEFAULT_WEIGHTS)     # edit to re-weight; missing components are renormalised
MIN_COVERAGE = 0.5

print(f"Loaded {len(clinical_evidence_df):,} source rows")
print(f"Evaluation run: {EVAL_RUN_ID}")
print(f"Weights: {SCORE_WEIGHTS}")

In [ ]:
score_check_results = check_clinical_evidence_score(clinical_evidence_df)
score_check_summary = summarize_score_check(score_check_results)

print(score_check_summary)
score_check_failures = score_check_results[
    ~score_check_results["score_check_pass"]
]
score_check_failures[[
    "source_key",
    "source_score",
    "reasoning_final_score",
    "score_check_reason",
]]

In [ ]:
selected_rows = latest_rows(clinical_evidence_df, SELECTED_MOLECULES)
selected_rows[["drug_name", "created_at", "score"]]

In [ ]:
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

FRAME_KEYS = (
    "claim_rows", "claim_results", "judge_results", "context_precision_results",
    "field_relevance_results", "claim_relevance_rows", "claim_relevance_results",
)


def _stage(name, molecule_name, errors, function):
    """Run one eval stage; a failure is recorded and does not stop the other stages."""
    try:
        return function()
    except Exception as error:
        errors.append(f"{name}: {error}")
        print(f"Stage '{name}' failed for {molecule_name}: {error}")
        return pd.DataFrame()


def evaluate_molecule(selected_row):
    molecule_name = str(selected_row["drug_name"])
    molecule_rows = selected_row.to_frame().T
    reset_claude_call_stats()
    started_at = time.perf_counter()
    errors = []
    frames = {key: pd.DataFrame() for key in FRAME_KEYS}

    # Faithfulness / claim precision
    claim_rows = _stage("claim_verification", molecule_name, errors,
                        lambda: run_claim_verification(molecule_rows, model=EVAL_MODEL))
    frames["claim_rows"] = claim_rows
    frames["claim_results"] = summarize_claims(claim_rows)
    frames["judge_results"] = _stage("llm_judge", molecule_name, errors,
                                     lambda: run_llm_judge(molecule_rows, model=EVAL_MODEL))

    # Precision of the retrieved context
    frames["context_precision_results"] = _stage(
        "context_precision", molecule_name, errors,
        lambda: run_context_precision(molecule_rows, model=EVAL_MODEL, fetch_pages=FETCH_SOURCE_PAGES_FOR_PRECISION))

    # Relevance
    frames["field_relevance_results"] = _stage("field_relevance", molecule_name, errors,
                                               lambda: run_field_relevance(molecule_rows, model=EVAL_MODEL))
    frames["claim_relevance_rows"] = _stage("claim_relevance", molecule_name, errors,
                                            lambda: run_claim_relevance(claim_rows, molecule_rows, model=EVAL_MODEL))
    frames["claim_relevance_results"] = summarize_claim_relevance(frames["claim_relevance_rows"])

    print(
        f"{molecule_name}: {'failed stages: ' + '; '.join(errors) if errors else 'all stages ok'} "
        f"in {time.perf_counter() - started_at:.1f}s, usage={get_claude_call_stats()}"
    )
    return frames


collected = {key: [] for key in FRAME_KEYS}

with ThreadPoolExecutor(max_workers=1) as executor:  # raise only if your Claude quota allows parallel molecules
    futures = [executor.submit(evaluate_molecule, row) for _, row in selected_rows.iterrows()]
    for future in as_completed(futures):
        for key, frame in future.result().items():
            if not frame.empty:
                collected[key].append(frame)

(claim_rows, claim_results, judge_results, context_precision_results,
 field_relevance_results, claim_relevance_rows, claim_relevance_results) = (
    pd.concat(collected[key], ignore_index=True) if collected[key] else pd.DataFrame() for key in FRAME_KEYS
)

print(f"Completed evaluations for {len(selected_rows)} molecule(s)")

In [ ]:
eval_results = build_eval_scores(
    score_check_results,
    claim_results,
    judge_results,
    context_precision_results,
    field_relevance_results,
    claim_relevance_results,
    amber_threshold=65.0,
    green_threshold=85.0,
    weights=SCORE_WEIGHTS,
    min_coverage=MIN_COVERAGE,
)

summary_columns = [
    "source_key",
    "source_score",
    "score_check_pass",
    "claim_precision_score",
    "judge_faithfulness_score",
    "context_precision_score",
    "field_relevance_score",
    "claim_relevance_score",
    "composite_coverage",
    "composite_score",
    "eval_status",
]
eval_results[[column for column in summary_columns if column in eval_results.columns]]

In [ ]:
# Drill-down: where precision and relevance are lost

# 1. Claims that are not supported by the retrieved sources or pipeline data (hallucination risk)
if not claim_rows.empty:
    display_columns = [c for c in ["field", "claim", "verdict", "evidence_basis", "evidence"] if c in claim_rows.columns]
    display(claim_rows.loc[claim_rows["verdict"] != "supported", display_columns])

# 2. Precision of the retrieved sources, per pipeline step
if not context_precision_results.empty and "context_precision_by_step" in context_precision_results.columns:
    for _, result_row in context_precision_results.iterrows():
        print(result_row["drug_name"], "overall:", result_row.get("context_precision_score"))
        by_step = result_row.get("context_precision_by_step")
        if isinstance(by_step, dict):
            display(pd.DataFrame(by_step).T)
        details = result_row.get("context_details")
        if isinstance(details, list):
            display(pd.DataFrame(details).query("verdict != 'relevant'")[["title", "domain", "steps", "verdict", "reason"]])

# 3. Relevance per report section
relevance_columns = [c for c in field_relevance_results.columns if c.startswith("relevance_") and c != "relevance_details"]
if relevance_columns:
    display(field_relevance_results[["drug_name", *relevance_columns, "field_relevance_mean", "fields_not_drug_specific"]])

# 4. Claims that do not bear on the evidence score
if not claim_relevance_rows.empty:
    display(claim_relevance_rows.loc[claim_relevance_rows["claim_relevance"] != "direct", ["field", "claim", "claim_relevance"]])

In [ ]:
import json
import pandas as pd

output_dir = repo_root / "medical_potential" / "clinical_evidence" / "evals"
excel_output_path = output_dir / f"clinical_evidence_evaluation_results_{pd.Timestamp.now():%Y%m%d_%H%M%S}.xlsx"


def _excel_safe(dataframe):
    """Excel cannot store lists/dicts or tz-aware datetimes."""
    export_dataframe = dataframe.copy()
    if export_dataframe.columns.duplicated().any():
        duplicates = sorted(set(export_dataframe.columns[export_dataframe.columns.duplicated()]))
        raise ValueError(f"Duplicate column names before Excel export: {duplicates}")
    for column in export_dataframe.select_dtypes(include=["datetimetz"]).columns:
        export_dataframe[column] = export_dataframe[column].dt.tz_localize(None)
    for column in export_dataframe.columns:
        if export_dataframe[column].map(lambda value: isinstance(value, (list, dict))).any():
            export_dataframe[column] = export_dataframe[column].map(
                lambda value: json.dumps(value, default=str) if isinstance(value, (list, dict)) else value
            )
    return export_dataframe


def _join(parts):
    return " | ".join(part for part in parts if part)


def _field_rationales(details):
    entries = (details or {}).get("fields", []) if isinstance(details, dict) else []
    return _join(
        f"{e.get('field', '').removeprefix('report_')}: {e.get('score')}/5 - {e.get('rationale', '')}"
        for e in entries if isinstance(e, dict)
    )


def build_eval_metrics(overall, judge, context, field_rel, claim_rel_rows):
    """One row per molecule; every score is 0-1."""
    records = []
    for _, row in overall.iterrows():
        key = row["source_key"]

        def pick(frame):
            if frame is None or frame.empty or "source_key" not in frame:
                return {}
            match = frame[frame["source_key"] == key]
            return match.iloc[0].to_dict() if len(match) else {}

        j, c, f = pick(judge), pick(context), pick(field_rel)

        # Faithfulness: LLM judge, overall rating 1-5 rescaled to 0-1.
        faith = row.get("judge_faithfulness_score")
        faith_rationale = _join([
            j.get("faithfulness_score_rationale"),
            f"Score supported: {j.get('score_supported')} - {j.get('score_supported_rationale')}" if j else "",
            f"Why not higher valid: {j.get('why_not_higher_valid')} - {j.get('why_not_higher_valid_rationale')}" if j else "",
            f"Internal consistency: {j.get('internal_consistency')} - {j.get('internal_consistency_rationale')}" if j else "",
        ])

        # Relevance: mean of field relevance and claim relevance (whichever are available).
        field_score, claim_score = row.get("field_relevance_score"), row.get("claim_relevance_score")
        parts = [v for v in (field_score, claim_score) if pd.notna(v)]
        relevance = sum(parts) / len(parts) if parts else None
        claim_text = ""
        if claim_rel_rows is not None and not claim_rel_rows.empty:
            labels = claim_rel_rows.loc[claim_rel_rows["source_key"] == key, "claim_relevance"].value_counts()
            claim_text = (
                f"Claims: {labels.get('direct', 0)} direct, {labels.get('contextual', 0)} contextual, "
                f"{labels.get('irrelevant', 0)} irrelevant (score {claim_score:.2f})" if pd.notna(claim_score) else ""
            )
        relevance_rationale = _join([
            f"Field relevance {field_score:.2f}: " + _field_rationales(f.get("relevance_details")) if pd.notna(field_score) else "",
            claim_text,
        ])

        # Precision: share of retrieved sources judged relevant.
        precision = row.get("context_precision_score")
        irrelevant = [
            f"{d.get('title') or d.get('url')}: {d.get('reason')}"
            for d in (c.get("context_details") or []) if isinstance(d, dict) and d.get("verdict") != "relevant"
        ]
        precision_rationale = _join([
            f"{c.get('sources_relevant')} of {c.get('sources_total')} retrieved sources relevant"
            f" ({c.get('sources_irrelevant')} irrelevant, {c.get('sources_cannot_tell')} cannot tell)" if c.get("sources_total") else c.get("context_error", ""),
            "Not relevant: " + "; ".join(irrelevant[:10]) if irrelevant else "",
        ])

        records.append({
            "Drug Name": row.get("drug_name") or row.get("source_key", "").split("|")[0],
            "Faithfulness Score": faith,
            "Faithfulness Rationale": faith_rationale,
            "Relevance Score": relevance,
            "Relevance Rationale": relevance_rationale,
            "Precision Score": precision,
            "Precision Rationale": precision_rationale,
        })
    return pd.DataFrame(records)


eval_metrics = build_eval_metrics(
    eval_results, judge_results, context_precision_results, field_relevance_results, claim_relevance_rows
)

with pd.ExcelWriter(excel_output_path, engine="openpyxl") as writer:
    for dataframe, sheet_name in [
        (score_check_results, "score_check_results"),
        (claim_rows, "claim_results"),
        (judge_results, "judge_results"),
        (context_precision_results, "context_precision"),
        (field_relevance_results, "field_relevance"),
        (claim_relevance_rows, "claim_relevance"),
        (eval_results, "overall_results"),
        (eval_metrics, "eval metrics"),
    ]:
        if dataframe is not None and not dataframe.empty:
            _excel_safe(dataframe).to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved evaluation results to {excel_output_path}")

if PUSH_TO_BQ:
    push_eval_rows(build_bq_rows(eval_results.to_dict("records"), EVAL_RUN_ID, EVAL_MODEL))
    print("Pushed eval rows to BigQuery")